# MM4250: stuck check and thru check

**Stuck check**: ALL_OPEN, internal short, ALL_OPEN, internal load, ALL_OPEN, then each port in the cooldown file with ALL_OPEN after it. Every position is compared with its neighbours, so a channel that won't close (stuck open) or won't open (stuck closed) stands out. Run it on the way down in the 30-80 K range and first thing at any temperature where you'll take data.

**Thru check**: the direct test through the fridge's thru line (the file's `lines.thru`, wired to `thru_channel`); it asks you to move VNA port 1's cable and back.

**Gate test** (diagnosis, at the end): each HV gate line of a channel on its own, for a channel the stuck check can't explain.

- Only one notebook can hold the switch: run the other notebook's **Close** cell first.
- Nothing is saved unless you ask (`SAVE_DB`: the lab's `..._stuck` database; `SAVE_TOUCHSTONE`: files + `positions.csv` beside it). The switch is always left `ALL_OPEN`, even if a check fails.
- **If anything is stuck, stop**: don't keep switching it cold. Run Close and tell Charlie.

**Setup**: which cooldown file.

In [ ]:
COOLDOWN_FILE = "cooldowns/CD05.json"   # this cooldown's file (see README.md)

import sys
from pathlib import Path
try:
    here = Path(_dh[0])                      # this notebook's folder (Jupyter)
except NameError:
    here = Path.cwd()
root = here
while not (root / "drivers").is_dir() and root != root.parent:
    root = root.parent                       # the folder with drivers/ in it
sys.path[:0] = [str(here), str(root)]
%matplotlib inline

import vna, db, ecal, plots
from scripts import switch_vna as sv   # the lab's safe switch + VNA functions
cfg = db.load_cooldown(here / COOLDOWN_FILE)
db.print_cooldown(cfg)

**Connect**: VNA, switch, fridge (optional: without the VPN it warns and carries on).

In [ ]:
from qcodes.instrument import Instrument
from drivers.KeysightVNA_driver import KeysightP5004B
from drivers.MM4250_QCodes_driver import MM4250

Instrument.close_all()                       # so re-running this cell is safe
ksvna = KeysightP5004B("ksvna", "TCPIP0::QTSF-Measurement::hislip_PXI0_CHASSIS1_SLOT1_INDEX0::INSTR")
ksvna.output(False)                          # source off first: connecting the switch moves it
switch = MM4250("switch")                    # connecting sets ALL_OPEN

fridge = None                                # only for temperatures: optional
try:
    from maybell_qcodes import MaybellFridge
    fridge = MaybellFridge("fridge", host="10.179.200.168")   # needs the GlobalConnect VPN
except Exception as e:
    print(f"[warning] no fridge ({type(e).__name__}: {e}); carrying on without temperatures")
print("Fridge (mixing plate):", sv.format_kelvin(sv.read_temperatures(fridge)["mxp"]))

**Stuck check**: about a minute. VNA port 1 on the input line, port 2 on the return line, LNA on. `SWEEP_MODE` picks what each sweep measures: `"forward"` (default) is S11 and S21 with only VNA port 1 sourcing, so nothing is sent down the return line into the LNA's output; `"2port"` adds S12/S22 (port 2 sources too, at `POWER_DBM`). The checks only use S21, and those are the same numbers in both modes. Forward also sets port 2's source Off on the VNA itself (it stays Off afterwards), and refuses to sweep while another trace (e.g. S22) is on the VNA: delete it there, or run `sv.remove_traces(ksvna)`.

Reading it: each row's *contrast* says how far it stands from what it should differ from (a closed position from the ALL_OPEN before it; an ALL_OPEN from the position before it). Every contrast should be well above 0 dB. A closed position near 0 dB **didn't close** (stuck open); an ALL_OPEN near 0 dB **didn't open** after the position before it (stuck closed). Once one is stuck, later rows can look wrong too: the first one counts. With `stuck_check.pass_db` set in the cooldown file, rows below it are flagged; with `null` (no pass mark yet), nothing is flagged and you read the numbers. Bottom plot: median |S21 - first ALL_OPEN| per position (the old A1 bars; compare neighbours, the chain drifts while cooling).

In [ ]:
POINTS = 2001            # sweep points across the cooldown file's band
POWER_DBM = -20          # VNA source power on the input line, dBm (at most 0)
SWEEP_MODE = "forward"   # "forward": port 1 only, S11 + S21, nothing into the LNA output (safe default);
                         # "2port": all four, port 2 also sources at POWER_DBM down the return line
SAVE_DB = False          # True: every position into the "stuck" database
SAVE_TOUCHSTONE = False  # True: one .s2p per position + positions.csv in Sweeps/stuck_check/ beside it

STUCK_DB = db.open_database(cfg, "stuck", create=SAVE_DB)
save = db.Saver(STUCK_DB if SAVE_DB else None, "stuck_check", files=SAVE_TOUCHSTONE,
                files_root=STUCK_DB.parent)
stuck = vna.stuck_check(db.with_sweep(cfg, POINTS, POWER_DBM, SWEEP_MODE), ksvna, switch, fridge, save=save)      # ends ALL_OPEN
vna.print_check(stuck, cfg["stuck_check"]["pass_db"])
if SAVE_TOUCHSTONE:
    db.write_check_csv(stuck, save.folder)
if SAVE_DB:
    db.backup_db(STUCK_DB)
plots.plot_check(stuck, cfg["switch_serial"]);

**Thru check**: asks you to move VNA port 1's cable from the input line to the thru line (Enter; `n` stops before anything moves), closes and opens `thru_channel` twice, then asks you to move the cable back. S21 is then a transmission: high with the channel closed, down at the switch's isolation with it open. Compared with `stuck_check.thru_pass_db` (20 dB is a guess; the warm run shows the real levels). **Move the cable back** when it asks: takes and the stuck check need port 1 on the input line.

In [ ]:
POINTS = 2001            # sweep points across the cooldown file's band
POWER_DBM = -40          # lower than the stuck check: the thru line is less attenuated, so -20 dBm
                         # would put the LNA near compression (Charlie's notes, 2026-11-09)
SWEEP_MODE = "forward"   # "forward": port 1 only, S11 + S21, nothing into the LNA output (safe default);
                         # "2port": all four, port 2 also sources at POWER_DBM down the return line
SAVE_DB = False          # True: every position into the "stuck" database
SAVE_TOUCHSTONE = False  # True: one .s2p per position + positions.csv in Sweeps/thru_check/ beside it

STUCK_DB = db.open_database(cfg, "stuck", create=SAVE_DB)
save = db.Saver(STUCK_DB if SAVE_DB else None, "thru_check", files=SAVE_TOUCHSTONE,
                files_root=STUCK_DB.parent)
thru = vna.thru_check(db.with_sweep(cfg, POINTS, POWER_DBM, SWEEP_MODE), ksvna, switch, fridge, save=save)        # asks about the cable; ends ALL_OPEN
if thru is not None:
    vna.print_check(thru)
    if SAVE_TOUCHSTONE:
        db.write_check_csv(thru, save.folder)
    if SAVE_DB:
        db.backup_db(STUCK_DB)
    plots.plot_check(thru, cfg["switch_serial"]);

**Hold the thru line open**: connects the thru channel to RFC and leaves it, so you can use the VNA from its own software (e.g. to check the HEMT). Meanwhile, every `INTERVAL_S` (10 s) it reads every fridge thermometer (MXP, MXP2, STP, CMN, STP2, CFP, PRP): each reading goes into the CSV as it's taken (`SAVE_CSV`: the cooldown folder's `Sweeps/thru_hold/<time>_temps.csv`), and the table under this cell updates in place with the newest ones. A thermometer that can't be read shows `--` (empty in the CSV). Run the next cell when you're done.

Power dissipated at the switch while it's held:
- Gate (holding it closed): under 1 uW (Menlo datasheet)
- RF, source off: 0
- RF, source on: about 7 pW at -40 dBm from the VNA, about 0.7 nW at -20 dBm

In [ ]:
from datetime import datetime
INTERVAL_S = 10    # seconds between fridge readings
SAVE_CSV = True    # True: each reading appended to Sweeps/thru_hold/<time>_temps.csv as it's taken
SHOW_LAST = 6      # readings shown in the live table under this cell

if globals().get("hold") is not None:
    sv.release(switch, ksvna, hold)      # a hold from an earlier run of this cell
csv_path = (db.open_database(cfg, "vna", create=False).parent / "Sweeps" / "thru_hold"
            / f"{datetime.now():%Y%m%d_%H%M%S}_temps.csv")
ksvna.output(False)                      # leave the source off while it's held
hold = sv.hold_state(switch, ksvna, cfg["thru_channel"], fridge=fridge, interval_s=INTERVAL_S,
                     csv_path=csv_path, save_csv=SAVE_CSV, show_last=SHOW_LAST)

**Open it again**: back to ALL_OPEN, stops the fridge logger (after the reading in progress) and prints how many readings it took, the last one, and where the CSV is.

In [ ]:
sv.release(switch, ksvna, hold)          # ALL_OPEN first, then stops the log and prints a summary
hold = None

**Gate test (diagnosis)**: each RF channel closes 4 HV gate lines (driver `STATE_HV_PINS`; RF4 = HV2 + HV8, shared with RF3 and RF5, plus its own HV10 + HV16). The stuck check only sees whole channels, so it can miss a single gate: one frozen **closed** behind a gate that opens, or one of a channel's own gates that won't close while the rest do (what RF4 looked like on 11/11 and 11/24: the reflection stops ~0.12 ns before its connector).

This sends, for each channel in `CHANNELS`: each of its gate lines alone, its shared lines, and shared + each own line, then the channel the normal way, each with ALL_OPEN after it, and the internal standards at both ends. These combinations are **not in Menlo's state table** (each line is still 0 V or ~90 V, like a normal state; ask Menlo first). Off unless you set `RUN_GATE_TEST = True` (so Run All never starts it). RF source off for every move; it asks before anything moves and always ends ALL_OPEN. The bytes are rebuilt from the driver's own table and checked against Menlo's states before anything is sent. Saved to its own database (`..._Cooldown_<NN>_hv_lines.db`) and `Sweeps/hv_lines/` beside it.

Reading it: the first table is each position's contrast against the ALL_OPEN before it (like the stuck check). The second says where each position reflects, one way: *from internal stds* (no NIST definitions: ALL_OPEN is ~0, an RF connector is ~+0.25 ns, from the warm 9/25 data) and *from connector* (NIST; ~0 = at the connector, negative = inside the switch). Only rows near 0 dB mean anything there (a load's phase is mostly noise).
- A gate on its own should look like ALL_OPEN (contrast ~0 dB, ~0 ns from the internal stds): it needs the gates in front of it, or the one it's paired with, to close too. If one alone doesn't, a gate it needs is frozen closed (e.g. HV8 alone moving the reflection out means HV2 is stuck closed).
- A gate that does nothing leaves the reflection where it was without it: compare shared vs shared + each own line vs the full channel.
- Then stop and send the numbers to Charlie: don't keep switching a gate that looks stuck.

In [ ]:
RUN_GATE_TEST = False    # True to run it: these gate combinations aren't in Menlo's state table
CHANNELS = [4]           # channels whose gates to test; e.g. [3, 4, 5] adds the two that share RF4's first gates
POINTS = 2001            # sweep points across the cooldown file's band
POWER_DBM = -20          # VNA source power on the input line, dBm (at most 0)
SWEEP_MODE = "forward"   # as in the stuck check
SAVE_DB = True           # True: every position into its own database, ..._hv_lines
SAVE_TOUCHSTONE = True   # True: one .s2p per position + positions.csv in Sweeps/hv_lines/ beside it
DEFINITIONS = None       # NIST set for the "from connector" column: None = from the temperature, or "3K" / "295K"

hv = None
if not RUN_GATE_TEST:
    print("Gate test not run (RUN_GATE_TEST = False): nothing moved, nothing saved.")
else:
    HV_DB = db.open_database(cfg, "hv_lines", create=SAVE_DB)
    save = db.Saver(HV_DB if SAVE_DB else None, "hv_lines", files=SAVE_TOUCHSTONE,
                    files_root=HV_DB.parent)
    hv = vna.hv_test(db.with_sweep(cfg, POINTS, POWER_DBM, SWEEP_MODE), ksvna, switch, fridge,
                     channels=CHANNELS, save=save)   # asks first; ends ALL_OPEN
if hv is not None:
    vna.print_check(hv)                              # contrast vs the ALL_OPEN before each position
    if SAVE_TOUCHSTONE:
        db.write_check_csv(hv, save.folder)
    if SAVE_DB:
        db.backup_db(HV_DB)
    print()
    hv_pos = ecal.hv_positions(hv, DEFINITIONS)      # where each position reflects
    plots.plot_check(hv, cfg["switch_serial"])
    plots.plot_hv_test(hv, hv_pos);

**Your own HV lines (round-trip timing)**: turns on exactly the HV lines you list, one set per measurement, with ALL_OPEN after each and the internal standards at both ends, then says how far out each one reflects (round trip and one way, ns). Same safety as the gate test: off unless `RUN_MY_LINES = True`, asks before anything moves, RF source off for every move, ends ALL_OPEN, bytes checked against Menlo's states first, and only lines some Menlo state uses (HV4-6 and HV22 are refused). Not in Menlo's state table: ask Menlo first.

`CHANNEL` is the channel the lines belong to: it labels the runs and picks NIST's port for the *from connector* column. *From internal stds* needs no NIST: ALL_OPEN is ~0, an RF connector ~+0.5 ns round trip (warm 9/25). Saved like the gate test (`..._hv_lines` database, `Sweeps/hv_lines/`).

In [ ]:
RUN_MY_LINES = False     # True to run it: these combinations aren't in Menlo's state table
CHANNEL = 4              # the channel these lines belong to (labels + NIST port for "from connector")
MY_LINES = [             # one measurement per entry: the HV lines that are on together
    ["HV2", "HV8"],
    ["HV2", "HV8", "HV10"],
    ["HV2", "HV8", "HV16"],
]
ALSO_FULL = True         # also CHANNEL the normal way (all its lines), to compare with
POINTS = 2001            # sweep points across the cooldown file's band
POWER_DBM = -20          # VNA source power on the input line, dBm (at most 0)
SWEEP_MODE = "forward"   # as in the stuck check
SAVE_DB = True           # True: every position into the ..._hv_lines database
SAVE_TOUCHSTONE = True   # True: one .s2p per position + positions.csv in Sweeps/hv_lines/
DEFINITIONS = None       # NIST set for "from connector": None = from the temperature, or "3K" / "295K"

mine = None
if not RUN_MY_LINES:
    print("Not run (RUN_MY_LINES = False): nothing moved, nothing saved.")
else:
    HV_DB = db.open_database(cfg, "hv_lines", create=SAVE_DB)
    save = db.Saver(HV_DB if SAVE_DB else None, "hv_lines", files=SAVE_TOUCHSTONE,
                    files_root=HV_DB.parent)
    mine = vna.hv_test(db.with_sweep(cfg, POINTS, POWER_DBM, SWEEP_MODE), ksvna, switch, fridge,
                       channels=[CHANNEL], combos=MY_LINES, full=ALSO_FULL, save=save)   # asks first; ends ALL_OPEN
if mine is not None:
    vna.print_check(mine)                            # contrast vs the ALL_OPEN before each position
    if SAVE_TOUCHSTONE:
        db.write_check_csv(mine, save.folder)
    if SAVE_DB:
        db.backup_db(HV_DB)
    print()
    mine_pos = ecal.hv_positions(mine, DEFINITIONS)  # round trip + one way, from the internal standards and the connector
    plots.plot_hv_test(mine, mine_pos);

**Close**: before anyone else needs the switch or VNA. Backs up the stuck database if there is one, leaves the switch `ALL_OPEN` and releases everything.

In [ ]:
STUCK_DB = db.open_database(cfg, "stuck", create=False)
if STUCK_DB.is_file():
    db.backup_db(STUCK_DB)
sv.all_open(switch, ksvna)                    # source off for the move
print("switch, last sent:", switch.state.cache.get(get_if_invalid=False))   # not read back: the lab driver's readback lags
ksvna.output(False)
ksvna.close()
switch.close()
if fridge is not None:
    fridge.close()